# E-commerce Fraud Data EDA

Task 1 notebook for cleaning checks, class imbalance review, IP-to-country enrichment, and fraud pattern exploration.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import FRAUD_DATA, IP_COUNTRY
from src.data_utils import basic_clean, class_distribution, load_fraud_data, load_ip_country
from src.features import add_country_by_ip, engineer_fraud_features


In [ ]:
fraud_raw = load_fraud_data(FRAUD_DATA)
ip_ranges = load_ip_country(IP_COUNTRY)
fraud_raw.head()

In [ ]:
cleaning_summary = {
    'rows_before': len(fraud_raw),
    'duplicates': int(fraud_raw.duplicated().sum()),
    'missing_values': fraud_raw.isna().sum().to_dict(),
}
cleaning_summary

In [ ]:
fraud = engineer_fraud_features(add_country_by_ip(basic_clean(fraud_raw), ip_ranges))
class_distribution(fraud, 'class')

In [ ]:
fraud[['purchase_value', 'age', 'time_since_signup_hours', 'hour_of_day', 'day_of_week']].describe()

In [ ]:
fraud.groupby('class')[['purchase_value', 'age', 'time_since_signup_hours', 'user_transaction_count', 'device_transaction_count']].median()

In [ ]:
country_fraud = (
    fraud.groupby('country')['class']
    .agg(transactions='count', fraud_rate='mean', fraud_cases='sum')
    .query('transactions >= 20')
    .sort_values(['fraud_rate', 'transactions'], ascending=[False, False])
)
country_fraud.head(15)

Document the main EDA takeaways in `reports/interim_report_task1.md`: class imbalance, fraud-heavy countries, suspicious signup-to-purchase timing, and any missing-value decisions.